# FASTA 데이터 탐색

당시 실험을 보존한 참고 노트북입니다. 실제 데이터와 임베딩·GO 매핑은 별도로 준비해야 합니다. 현재 기본 실행 경로는 scripts/cafa.py입니다. 원본에 남은 실험 설정·최종 요약은 최종 제출 모델을 의미하지 않습니다.


# FASTA 읽기 
FASTA는 단백질 서열들이 들어있는 파일이고, 순수하게 아미노산 문자열

위 파일을 통해 단백질 ID, 단백질 길이 등을 알수있음

### 줄별로 뜯는거

In [ ]:
# 줄별로 뜯는거

from pathlib import Path
import numpy as np

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
fasta_path = ROOT / "data" / "raw" / "query.fasta"

seq_lengths = []
current_seq = []
protein_count = 0

with open(fasta_path, "r") as f:
    for line in f:
        line = line.strip()

        # 헤더 줄
        if line.startswith(">"):
            if current_seq:
                seq_lengths.append(len("".join(current_seq)))
                current_seq = []
                protein_count += 1
        else:
            # 서열 줄
            current_seq.append(line)

    # 마지막 단백질 처리
    if current_seq:
        seq_lengths.append(len("".join(current_seq)))
        protein_count += 1


### 숫자로 요약하는거

In [ ]:
# 숫자로 요약하는거

import numpy as np
seq_lengths = np.array(seq_lengths)

print("단백질 개수:", protein_count)
print("최소 길이:", seq_lengths.min())                 # 가장 짧은 단백질 길이
print("중앙값 길이:", np.median(seq_lengths))          # 길이를 정렬했을 때 가운데 값
print("평균 길이:", seq_lengths.mean())                # 전체 길이 평균
print("상위 10% 길이(p90):", np.percentile(seq_lengths, 90))  # 긴 단백질 기준(상위 10%)
print("상위 5% 길이(p95):", np.percentile(seq_lengths, 95))   # 긴 단백질 기준(상위 5%)
print("상위 1% 길이(p99):", np.percentile(seq_lengths, 99))   # 매우 긴 단백질 기준(상위 1%)
print("최대 길이:", seq_lengths.max())                 # 가장 긴 단백질 길이


### 위험 구간 비율

In [ ]:
thresholds = [512, 1024, 2048, 4096]

for t in thresholds:
    ratio = (seq_lengths > t).mean() * 100
    print(f"길이 {t} 초과 비율: {ratio:.2f}%")   # 특정 길이보다 긴 단백질이 얼마나 되는지
